In [ ]:
import os
import numpy as np
import pandas as pd
import pyodbc
from collections import Counter
from dataclasses import dataclass

# -----------------------------
# Config conexión
# -----------------------------
SERVER = "10.147.17.185"
PORT = 1433
USERNAME = "cmpcuser"
PASSWORD_ENV_VAR = "OTMS_SQL_PASSWORD_DEV"
PASSWORD = os.getenv(PASSWORD_ENV_VAR)
DRIVER = "{ODBC Driver 17 for SQL Server}"

DB = "cmpc_20240925_093000"
SCHEMA = "dbo"
TABLE = "ypf_alarms"

TIME_COL = "ALARMDATETIME"          # ya confirmado
TAG_COL = "TAG_DESCRIPTION"         # puedes cambiar a ALARM_ID si prefieres
MSG_COL = "ALARM_DESCRIPTION"       # texto/mensaje
PRIORITY_COL = "PRIORITY"           # opcional (no usado en rules por ahora)

# -----------------------------
# Config de bloques + streaming
# -----------------------------
GAP_MIN = 5
CHUNKSIZE = 200_000

# -----------------------------
# Reglas Flood D (iniciales)
# Ajustables después de ver distribución de diversidad
# -----------------------------
DOMINANT_SHARE_TH = 0.60     # repetición fuerte (tag o msg)
LOW_UNIQUE_TAGS_TH = 10      # poca diversidad (ajustable)
MIN_BLOCK_N = 50             # no nos interesa flood en bloques pequeños

# -----------------------------
# Helpers
# -----------------------------
def connect():
    if not PASSWORD:
        raise RuntimeError(
            f"No encontré {PASSWORD_ENV_VAR}. "
            f"En PowerShell: setx {PASSWORD_ENV_VAR} \"TU_PASSWORD\" y reinicia la terminal."
        )
    conn_str = (
        f"DRIVER={DRIVER};"
        f"SERVER={SERVER},{PORT};"
        f"DATABASE={DB};"
        f"UID={USERNAME};"
        f"PWD={PASSWORD};"
        "TrustServerCertificate=yes;"
    )
    print(f"Conectando a {SERVER}:{PORT} | DB={DB} | user={USERNAME} ...")
    return pyodbc.connect(conn_str)

def pct_dict(x: np.ndarray, ps=(50,75,90,95,97,98,99,99.5,99.9)):
    return {f"p{str(p).replace('.','_')}": float(np.percentile(x, p)) for p in ps}

def safe_str(x):
    # Normaliza para Counter: evita None/NaN
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x)

# -----------------------------
# 1) Baseline global: alarmas por minuto
# -----------------------------
def compute_global_minute_percentiles(conn):
    full_table = f"[{DB}].[{SCHEMA}].[{TABLE}]"
    q = f"""
    SELECT
        DATEADD(minute, DATEDIFF(minute, 0, [{TIME_COL}]), 0) AS [minute],
        COUNT(1) AS alarms
    FROM {full_table}
    GROUP BY DATEADD(minute, DATEDIFF(minute, 0, [{TIME_COL}]), 0)
    """
    df_minute = pd.read_sql(q, conn)
    x = df_minute["alarms"].to_numpy()
    out = pct_dict(x)
    out["max_per_minute"] = int(x.max())
    return out

# -----------------------------
# 2) Streaming blocks + features
# -----------------------------
@dataclass
class BlockAgg:
    start: pd.Timestamp = None
    end: pd.Timestamp = None
    n: int = 0

    # per-minute inside block
    cur_minute: pd.Timestamp = None
    cur_minute_count: int = 0
    max_rate: int = 0

    # distributions
    tag_counts: Counter = None
    msg_counts: Counter = None

    def __post_init__(self):
        if self.tag_counts is None:
            self.tag_counts = Counter()
        if self.msg_counts is None:
            self.msg_counts = Counter()

def finalize_block(blk: BlockAgg):
    if blk.n == 0:
        return None

    duration_min = (blk.end - blk.start).total_seconds() / 60.0 if blk.end and blk.start else 0.0
    mean_rate = (blk.n / duration_min) if duration_min > 0 else float("inf")

    unique_tags = len(blk.tag_counts)
    unique_msgs = len(blk.msg_counts)

    top_tag = blk.tag_counts.most_common(1)[0][1] if unique_tags > 0 else 0
    top_msg = blk.msg_counts.most_common(1)[0][1] if unique_msgs > 0 else 0

    dominant_tag_share = top_tag / blk.n if blk.n > 0 else 0.0
    dominant_msg_share = top_msg / blk.n if blk.n > 0 else 0.0

    return {
        "start": blk.start,
        "end": blk.end,
        "n": blk.n,
        "duration_min": duration_min,
        "mean_rate": mean_rate,
        "max_rate": blk.max_rate,
        "unique_tags": unique_tags,
        "dominant_tag_share": dominant_tag_share,
        "unique_msgs": unique_msgs,
        "dominant_msg_share": dominant_msg_share,
    }

def compute_block_features(conn):
    full_table = f"[{DB}].[{SCHEMA}].[{TABLE}]"
    q = f"""
    SELECT
        [{TIME_COL}] AS t,
        [{TAG_COL}] AS tag,
        [{MSG_COL}] AS msg
    FROM {full_table}
    ORDER BY [{TIME_COL}] ASC
    """

    results = []
    blk = BlockAgg()
    last_t = None

    for chunk in pd.read_sql(q, conn, chunksize=CHUNKSIZE):
        if chunk.empty:
            continue

        chunk["t"] = pd.to_datetime(chunk["t"], errors="coerce")
        chunk = chunk.dropna(subset=["t"])
        if chunk.empty:
            continue

        for t, tag, msg in zip(chunk["t"], chunk["tag"], chunk["msg"]):
            # inicio del primer bloque
            if last_t is None:
                blk.start = t
                blk.end = t
                blk.n = 0
                blk.cur_minute = t.floor("min")
                blk.cur_minute_count = 0
                blk.max_rate = 0
                blk.tag_counts.clear()
                blk.msg_counts.clear()

            # ¿nuevo bloque por gap?
            gap_sec = (t - last_t).total_seconds() if last_t is not None else 0
            if last_t is not None and gap_sec > GAP_MIN * 60:
                # cerrar el minuto en curso antes de finalizar
                blk.max_rate = max(blk.max_rate, blk.cur_minute_count)
                row = finalize_block(blk)
                if row is not None:
                    results.append(row)

                # iniciar nuevo bloque
                blk = BlockAgg()
                blk.start = t
                blk.end = t
                blk.cur_minute = t.floor("min")
                blk.cur_minute_count = 0

            # actualizar conteos
            blk.end = t
            blk.n += 1
            blk.tag_counts[safe_str(tag)] += 1
            blk.msg_counts[safe_str(msg)] += 1

            # actualizar conteo por minuto
            m = t.floor("min")
            if blk.cur_minute is None:
                blk.cur_minute = m
                blk.cur_minute_count = 1
            elif m == blk.cur_minute:
                blk.cur_minute_count += 1
            else:
                # minuto cambió: actualiza max y resetea
                blk.max_rate = max(blk.max_rate, blk.cur_minute_count)
                blk.cur_minute = m
                blk.cur_minute_count = 1

            last_t = t

    # cerrar último bloque
    if blk.n > 0:
        blk.max_rate = max(blk.max_rate, blk.cur_minute_count)
        row = finalize_block(blk)
        if row is not None:
            results.append(row)

    return pd.DataFrame(results)

# -----------------------------
# 3) Flood classification (D)
# -----------------------------
def classify_floods(df_blocks: pd.DataFrame, minute_pcts: dict):
    p99 = minute_pcts["p99"]
    p99_9 = minute_pcts["p99_9"]

    # señales
    density_extreme = df_blocks["max_rate"] >= p99
    density_severe = df_blocks["max_rate"] >= p99_9

    repetition = (df_blocks["dominant_tag_share"] >= DOMINANT_SHARE_TH) | (df_blocks["dominant_msg_share"] >= DOMINANT_SHARE_TH)
    low_diversity = df_blocks["unique_tags"] <= LOW_UNIQUE_TAGS_TH

    big_enough = df_blocks["n"] >= MIN_BLOCK_N

    # Flood D: (densidad extrema) + (repetición o baja diversidad) + tamaño mínimo
    df_blocks["flood_candidate"] = big_enough & density_extreme & (repetition | low_diversity)

    # severidad
    df_blocks["flood_severity"] = "none"
    df_blocks.loc[df_blocks["flood_candidate"] & ~density_severe, "flood_severity"] = "medium"
    df_blocks.loc[df_blocks["flood_candidate"] & density_severe, "flood_severity"] = "severe"

    # una etiqueta “mild” opcional: densidad alta pero no extrema (p95 a p99) y repetición
    # (solo para diagnóstico, no para acción automática todavía)
    # df_blocks["mild_candidate"] = big_enough & (df_blocks["max_rate"] >= minute_pcts["p95"]) & ~density_extreme & repetition

    return df_blocks

# -----------------------------
# Run
# -----------------------------
def main():
    conn = connect()

    print("\nCalculando baseline global de alarmas/min ...")
    minute_pcts = compute_global_minute_percentiles(conn)
    print(pd.Series(minute_pcts).to_string())

    print("\nConstruyendo features por bloque (streaming) ...")
    df_blocks = compute_block_features(conn)

    print("\nClasificando floods (definición D inicial) ...")
    df_blocks = classify_floods(df_blocks, minute_pcts)

    print("\nResumen bloques:")
    print(df_blocks[["n","duration_min","mean_rate","max_rate","unique_tags","dominant_tag_share","unique_msgs","dominant_msg_share"]]
          .describe(percentiles=[0.5,0.75,0.9,0.95,0.99]).to_string())

    print("\nTop 20 por max_rate (minuto más intenso dentro del bloque):")
    print(df_blocks.sort_values("max_rate", ascending=False).head(20)[
        ["start","end","n","duration_min","max_rate","unique_tags","dominant_tag_share","unique_msgs","dominant_msg_share","flood_candidate","flood_severity"]
    ].to_string(index=False))

    print("\nTop 20 floods candidatos:")
    floods = df_blocks[df_blocks["flood_candidate"]].sort_values(["flood_severity","max_rate","n"], ascending=[False,False,False]).head(20)
    print(floods[
        ["start","end","n","duration_min","max_rate","unique_tags","dominant_tag_share","unique_msgs","dominant_msg_share","flood_severity"]
    ].to_string(index=False))

    # Guardar resultados
    out_dir = "artifacts"
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "blocks_features.parquet")
    df_blocks.to_parquet(out_path, index=False)
    print(f"\nGuardado: {out_path}")

    conn.close()

if __name__ == "__main__":
    main()